<hr style="border: 2px solid #7C6F64; margin-top: 10px;">

# Image Classification - DINOv3 Template

Template kompetisi image task. Alur notebook mengikuti referensi: instalasi, pustaka, seed,
konfigurasi, load train/test, EDA, validasi orientasi dan ukuran, training, post-train analysis,
serta pembuatan submission.


## Instalasi Modul

Jalankan cell ini sekali. Untuk Kaggle/Colab, sesuaikan instalasi `torch` dengan environment aktif.


In [ ]:
%pip install -q transformers accelerate timm scikit-learn matplotlib seaborn pillow opencv-python


## Pustaka


In [ ]:
import os, random, warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from transformers import AutoModel

warnings.filterwarnings("ignore")


## Seed


In [ ]:
def seed_everything(seed=42):
    import os
    import random

    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## Settings / Config


In [ ]:
class Settings:
    SEED = 42
    RUN_NAME = "classification_dinov3"

    # Ubah 4 path ini sesuai environment: local, Kaggle, atau Colab.
    DATA_ROOT = Path("/kaggle/input/your-competition")
    TRAIN_PATH = DATA_ROOT / "train"
    TEST_PATH = DATA_ROOT / "test"
    OUTPUT_PATH = Path("/kaggle/working")
    SAMPLE_SUBMISSION_PATH = DATA_ROOT / "sample_submission.csv"

    TRAIN_DIR = TRAIN_PATH
    TEST_DIR = TEST_PATH
    SAMPLE_SUBMISSION = SAMPLE_SUBMISSION_PATH

    MODEL_REGISTRY = {
        "dinov3_base": {
            "model_id": "facebook/dinov3-vitb16-pretrain-lvd1689m",
            "patch_size": 16,
            "img_size": (224, 224),
            "batch_size": 16,
            "lr": 2e-4,
            "freeze_backbone": True,
            "description": "DINOv3 ViT-B/16, backbone self-supervised ukuran base.",
            "why": "Representasi umum kuat dengan biaya GPU masih masuk akal.",
            "when_to_use": "Pakai sebagai default DINOv3 untuk dataset kecil-menengah atau eksperimen cepat.",
        },
        "dinov3_large": {
            "model_id": "facebook/dinov3-vitl16-pretrain-lvd1689m",
            "patch_size": 16,
            "img_size": (224, 224),
            "batch_size": 8,
            "lr": 1e-4,
            "freeze_backbone": True,
            "description": "DINOv3 ViT-L/16, kapasitas lebih besar dari base.",
            "why": "Fitur lebih kaya untuk domain sulit, namun VRAM dan waktu training naik.",
            "when_to_use": "Pakai jika base underfit atau validasi stagnan, dan GPU cukup longgar.",
        },
        "dinov3_huge": {
            "model_id": "facebook/dinov3-vith16plus-pretrain-lvd1689m",
            "patch_size": 16,
            "img_size": (224, 224),
            "batch_size": 4,
            "lr": 8e-5,
            "freeze_backbone": True,
            "description": "DINOv3 ViT-H+/16, backbone sangat besar untuk ekstraksi fitur kuat.",
            "why": "Cocok untuk mengejar skor akhir ketika fitur frozen sudah sangat informatif.",
            "when_to_use": "Pakai untuk final run, dataset visual kompleks, atau saat punya GPU besar.",
        },
    }

    # Ganti cukup key ini: "dinov3_base", "dinov3_large", atau "dinov3_huge".
    MODEL_NAME = "dinov3_base"
    MODEL_CONFIG = MODEL_REGISTRY[MODEL_NAME]
    MODEL_ID = MODEL_CONFIG["model_id"]
    MODEL_KWARGS = {"trust_remote_code": True}
    FREEZE_BACKBONE = MODEL_CONFIG.get("freeze_backbone", True)
    PATCH_SIZE = MODEL_CONFIG.get("patch_size", 16)

    OUTPUT_DIR = OUTPUT_PATH / f"{RUN_NAME}_{MODEL_NAME}"

    IMG_SIZE = MODEL_CONFIG.get("img_size", (224, 224))
    BATCH_SIZE = MODEL_CONFIG.get("batch_size", 16)
    NUM_WORKERS = 2
    EPOCHS = 8
    LR = MODEL_CONFIG.get("lr", 2e-4)
    WEIGHT_DECAY = 1e-4
    LABEL_SMOOTHING = 0.05
    GRAD_CLIP = 1.0
    USE_AMP = True

    VAL_SIZE = 0.2

    # True  : setiap subfolder train menjadi kelas sendiri.
    # False : binary classification; class di POSITIVE_CLASSES menjadi positive, sisanya negative.
    MULTICLASS = True
    CLASS_NAMES = None
    POSITIVE_CLASSES = ["positive", "pos", "real", "true", "1"]
    BINARY_CLASS_NAMES = ["negative", "positive"]

    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]


CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
seed_everything(CFG.SEED)

model_catalog_df = pd.DataFrame.from_dict(CFG.MODEL_REGISTRY, orient="index")
display(model_catalog_df[["model_id", "description", "why", "when_to_use", "img_size", "batch_size"]])
print("Selected model key:", CFG.MODEL_NAME)
print("Selected model id :", CFG.MODEL_ID)
print("Why              :", CFG.MODEL_CONFIG["why"])
print("When to use      :", CFG.MODEL_CONFIG["when_to_use"])



## Autentikasi Hugging Face


In [ ]:
try:
    from huggingface_hub import login

    hf_token = os.environ.get("HF_TOKEN")
    if hf_token is None and Path("/kaggle/working").exists():
        try:
            from kaggle_secrets import UserSecretsClient

            hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            hf_token = None

    if hf_token:
        login(token=hf_token)
        print("Hugging Face token aktif.")
    else:
        print("HF_TOKEN tidak ditemukan. Jika model gated, tambahkan token dulu.")
except Exception as e:
    print("Skip Hugging Face login:", repr(e))


## Load Train dan Test


In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def list_images(root_dir):
    root_dir = Path(root_dir)
    return sorted([p for p in root_dir.rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES])


def load_rgb_image(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    return img.convert("RGB")


def build_classification_items(train_dir, test_dir, class_names=None, multiclass=True, positive_classes=None, binary_class_names=None):
    train_dir, test_dir = Path(train_dir), Path(test_dir)
    raw_class_names = sorted([p.name for p in train_dir.iterdir() if p.is_dir()])
    if not raw_class_names:
        raise ValueError(f"Tidak menemukan subfolder kelas di {train_dir}")

    if multiclass:
        active_class_names = raw_class_names if class_names is None else list(class_names)
        label_to_idx = {name: i for i, name in enumerate(active_class_names)}
        missing = [name for name in active_class_names if name not in raw_class_names]
        if missing:
            raise ValueError(f"CLASS_NAMES tidak ditemukan di train_dir: {missing}")

        train_items = []
        for cls in active_class_names:
            for p in list_images(train_dir / cls):
                train_items.append((p, label_to_idx[cls]))
        return train_items, [(p, -1) for p in list_images(test_dir)], active_class_names, label_to_idx, raw_class_names

    binary_class_names = list(binary_class_names or ["negative", "positive"])
    if len(binary_class_names) != 2:
        raise ValueError("BINARY_CLASS_NAMES harus berisi tepat 2 label: [negative_label, positive_label]")

    positive_lookup = {str(x).lower() for x in (positive_classes or [])}
    matched_positive = [cls for cls in raw_class_names if cls.lower() in positive_lookup]
    if not matched_positive:
        raise ValueError(
            "MULTICLASS=False tetapi tidak ada folder yang cocok dengan POSITIVE_CLASSES. "
            f"Raw classes: {raw_class_names}. Ubah CFG.POSITIVE_CLASSES sesuai nama folder positif."
        )

    train_items = []
    for cls in raw_class_names:
        label = 1 if cls.lower() in positive_lookup else 0
        for p in list_images(train_dir / cls):
            train_items.append((p, label))

    label_to_idx = {binary_class_names[0]: 0, binary_class_names[1]: 1}
    return train_items, [(p, -1) for p in list_images(test_dir)], binary_class_names, label_to_idx, raw_class_names


def inspect_images(items):
    rows = []
    for p, y in tqdm(items, desc="inspect"):
        with Image.open(p) as img:
            exif_orientation = None
            try:
                exif = img.getexif()
                exif_orientation = exif.get(274, None)
            except Exception:
                pass
            w_raw, h_raw = img.size
        fixed = load_rgb_image(p)
        w_fixed, h_fixed = fixed.size
        rows.append(
            {
                "path": str(p),
                "filename": p.name,
                "label": int(y) if y is not None else -1,
                "width_raw": w_raw,
                "height_raw": h_raw,
                "width_fixed": w_fixed,
                "height_fixed": h_fixed,
                "aspect_ratio": w_fixed / max(h_fixed, 1),
                "orientation_exif": exif_orientation,
                "needs_exif_transpose": (w_raw, h_raw) != (w_fixed, h_fixed) or exif_orientation not in (None, 1),
            }
        )
    return pd.DataFrame(rows)


def show_class_samples(items, class_names, n_per_class=3):
    by_label = defaultdict(list)
    for p, y in items:
        by_label[y].append(p)
    rows = len(class_names)
    cols = n_per_class
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.2 * rows))
    axes = np.array(axes).reshape(rows, cols)
    for label_idx, cls in enumerate(class_names):
        paths = by_label[label_idx][:n_per_class]
        for j in range(cols):
            ax = axes[label_idx, j]
            ax.axis("off")
            if j < len(paths):
                ax.imshow(load_rgb_image(paths[j]))
                ax.set_title(f"{cls}\n{paths[j].name}", fontsize=9)
    plt.tight_layout()
    return fig


In [ ]:
train_items, test_items, class_names, label_to_idx, raw_class_names = build_classification_items(
    CFG.TRAIN_DIR,
    CFG.TEST_DIR,
    class_names=CFG.CLASS_NAMES,
    multiclass=CFG.MULTICLASS,
    positive_classes=CFG.POSITIVE_CLASSES,
    binary_class_names=CFG.BINARY_CLASS_NAMES,
)
CFG.CLASS_NAMES = class_names
CFG.NUM_CLASSES = len(class_names)
print("mode:", "multiclass" if CFG.MULTICLASS else "binary")
print("raw train folders:", raw_class_names)
print("active classes:", class_names)
print("label_to_idx:", label_to_idx)
print("train:", len(train_items), "test:", len(test_items))


## EDA dan Cek Orientasi / Ukuran


In [ ]:
train_meta = inspect_images(train_items)
test_meta = inspect_images(test_items)
display(train_meta.head())

print("Train size distribution")
display(train_meta.groupby(["width_fixed", "height_fixed"]).size().sort_values(ascending=False).head(20))
print("Test size distribution")
display(test_meta.groupby(["width_fixed", "height_fixed"]).size().sort_values(ascending=False).head(20))
print("EXIF orientation issues:", int(train_meta["needs_exif_transpose"].sum() + test_meta["needs_exif_transpose"].sum()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train_meta["aspect_ratio"], ax=axes[0], kde=True)
axes[0].set_title("Train aspect ratio")
sns.histplot(test_meta["aspect_ratio"], ax=axes[1], kde=True)
axes[1].set_title("Test aspect ratio")
plt.tight_layout()
plt.show()

class_counts = pd.Series([CFG.CLASS_NAMES[y] for _, y in train_items]).value_counts().reindex(CFG.CLASS_NAMES, fill_value=0)
display(class_counts.to_frame("count"))
class_counts.plot(kind="bar", figsize=(10, 4), title="Distribusi kelas")
plt.show()

show_class_samples(train_items, CFG.CLASS_NAMES, n_per_class=3)
plt.show()


## Dataset, Split, dan DataLoader


In [ ]:
class ImageClassificationDataset(Dataset):
    def __init__(self, items, img_size, train=False):
        self.items = list(items)
        self.img_size = tuple(img_size)
        self.train = train
        self.transform = self._build_transform()

    def _build_transform(self):
        transforms = [
            v2.ToImage(),
            v2.Resize(self.img_size, antialias=True),
        ]
        if self.train:
            transforms += [
                v2.RandomHorizontalFlip(p=0.5),
                v2.RandomApply([v2.ColorJitter(0.15, 0.15, 0.15, 0.05)], p=0.4),
            ]
        transforms += [
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=CFG.IMAGENET_MEAN, std=CFG.IMAGENET_STD),
        ]
        return v2.Compose(transforms)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = load_rgb_image(path)
        x = self.transform(img)
        return {"image": x, "label": torch.tensor(label, dtype=torch.long), "path": str(path)}


In [ ]:
print("Train/valid split memakai holdout biasa, bukan K-Fold. Untuk eksperimen ulang cukup ubah EPOCHS, MODEL_NAME, atau hyperparameter di Settings.")

labels = [y for _, y in train_items]
tr_items, va_items = train_test_split(
    train_items,
    test_size=CFG.VAL_SIZE,
    random_state=CFG.SEED,
    stratify=labels if min(pd.Series(labels).value_counts()) >= 2 else None,
)

train_ds = ImageClassificationDataset(tr_items, CFG.IMG_SIZE, train=True)
val_ds = ImageClassificationDataset(va_items, CFG.IMG_SIZE, train=False)
test_ds = ImageClassificationDataset(test_items, CFG.IMG_SIZE, train=False)

train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)

print(len(train_ds), len(val_ds), len(test_ds))


## Model DINOv3 + Classification Head


In [ ]:
class DinoV3Classifier(nn.Module):
    def __init__(self, model_name, num_classes, freeze_backbone=True):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name, **CFG.MODEL_KWARGS)
        hidden = self.backbone.config.hidden_size
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Dropout(0.2),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x):
        out = self.backbone(pixel_values=x)
        pooled = out.pooler_output if getattr(out, "pooler_output", None) is not None else out.last_hidden_state[:, 0]
        return self.head(pooled)


model = DinoV3Classifier(CFG.MODEL_ID, CFG.NUM_CLASSES, CFG.FREEZE_BACKBONE).to(DEVICE)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M parameters")
print(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, "M trainable")


## Train


In [ ]:
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, all_true, all_pred = 0.0, [], []

    pbar = tqdm(loader, desc="train" if is_train else "valid", leave=False)
    for batch in pbar:
        x = batch["image"].to(DEVICE)
        y = batch["label"].to(DEVICE)

        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=CFG.USE_AMP and DEVICE.type == "cuda"):
                logits = model(x)
                loss = criterion(logits, y)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None and CFG.USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
                    optimizer.step()

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        all_true.extend(y.detach().cpu().numpy().tolist())
        all_pred.extend(pred.detach().cpu().numpy().tolist())
        pbar.set_postfix(loss=loss.item())

    avg_loss = total_loss / max(len(loader.dataset), 1)
    acc = accuracy_score(all_true, all_pred)
    f1 = f1_score(all_true, all_pred, average="macro")
    return {"loss": avg_loss, "acc": acc, "f1_macro": f1}


def fit_classifier(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.USE_AMP and DEVICE.type == "cuda")

    best_score = -1
    history = []
    CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, CFG.EPOCHS + 1):
        train_log = run_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_log = run_one_epoch(model, val_loader, criterion)
        scheduler.step()

        row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_log.items()}, **{f"val_{k}": v for k, v in val_log.items()}}
        history.append(row)
        print(row)

        if val_log["f1_macro"] > best_score:
            best_score = val_log["f1_macro"]
            torch.save({"model": model.state_dict(), "cfg": CFG.__dict__, "score": best_score}, CFG.OUTPUT_DIR / "best_model.pt")
            print("Saved best:", best_score)

    return pd.DataFrame(history)


In [ ]:
history_df = fit_classifier(model, train_loader, val_loader)
history_df.tail()


## Post-Train Analysis


In [ ]:
history_df.to_csv(CFG.OUTPUT_DIR / "history.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric in zip(axes, ["loss", "acc", "f1_macro"]):
    ax.plot(history_df["epoch"], history_df[f"train_{metric}"], label="train")
    ax.plot(history_df["epoch"], history_df[f"val_{metric}"], label="val")
    ax.set_title(metric)
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

ckpt = torch.load(CFG.OUTPUT_DIR / "best_model.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
val_pred_df = predict_classifier(model, val_loader, class_names)
val_true = [class_names[y] for _, y in va_items]
print(classification_report(val_true, val_pred_df["label"], digits=4))

cm = confusion_matrix(val_true, val_pred_df["label"], labels=class_names)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.tight_layout()
plt.show()

hard_cases = val_pred_df.assign(true=val_true).query("label != true").sort_values("confidence", ascending=False)
hard_cases.head(20)


## Submission


In [ ]:
@torch.no_grad()
def predict_classifier(model, loader, class_names):
    model.eval()
    rows = []
    for batch in tqdm(loader, desc="predict"):
        x = batch["image"].to(DEVICE)
        logits = model(x)
        prob = torch.softmax(logits, dim=1).cpu().numpy()
        pred = prob.argmax(axis=1)
        for path, pidx, conf, probs in zip(batch["path"], pred, prob.max(axis=1), prob):
            row = {
                "id": Path(path).stem,
                "filename": Path(path).name,
                "label": class_names[int(pidx)],
                "label_idx": int(pidx),
                "confidence": float(conf),
            }
            for i, cls in enumerate(class_names):
                row[f"proba_{cls}"] = float(probs[i])
            rows.append(row)
    return pd.DataFrame(rows)


ckpt = torch.load(CFG.OUTPUT_DIR / "best_model.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
test_pred_df = predict_classifier(model, test_loader, class_names)


def align_to_sample_submission(pred_df, sample_path, target_candidates):
    sample_path = Path(sample_path)
    if not sample_path.exists():
        return pred_df[["id", target_candidates[0]]].copy()

    sample_df = pd.read_csv(sample_path)
    aligned = sample_df.copy()
    id_col = "id" if "id" in aligned.columns else aligned.columns[0]

    lookup_rows = []
    for row in pred_df.to_dict("records"):
        keys = []
        if "id" in row:
            keys.append(str(row["id"]))
        if "filename" in row:
            keys.extend([str(row["filename"]), Path(str(row["filename"])).stem])
        for key in dict.fromkeys(keys):
            rec = dict(row)
            rec["_key"] = key
            lookup_rows.append(rec)
    lookup = pd.DataFrame(lookup_rows).drop_duplicates("_key").set_index("_key")

    for col in aligned.columns:
        if col == id_col:
            continue
        source_col = col if col in pred_df.columns else next((c for c in target_candidates if c in pred_df.columns), None)
        if source_col is not None:
            aligned[col] = aligned[id_col].astype(str).map(lookup[source_col])
    return aligned

submission = align_to_sample_submission(test_pred_df, CFG.SAMPLE_SUBMISSION, target_candidates=("label",))
submission.to_csv(CFG.OUTPUT_DIR / "submission.csv", index=False)
test_pred_df.to_csv(CFG.OUTPUT_DIR / "test_predictions.csv", index=False)
submission.head()
